# 05 · Plot a model: plan views, sections and map layers

Loads a GoFEM model, puts it on the map, and draws depth slices (plan views) and vertical
sections, with your own map layers (rasters, depth surfaces, outlines, point tables) drawn with it.
The figures are matplotlib figures, so anything matplotlib can add, you can add.

* **Needs:** the model `.vtu` (GoFEM's output, with a `conductivity` array), its receiver file,
  and how to put it on the map: the model frame from notebook 02, or the stations' EDIs.
* **Writes:** figures in `work/05_figures/` (PNG; use `.pdf` or `.svg` in the file names for
  vector figures).
* **Takes:** under a minute for the sample.
* **GUI:** the plotting GUI (`mtplot`): *Model*, *Layers*, *Plan view*, *Sections* tabs.

The worked example plots the synthetic stand-in model from notebook 04. A model from the setup
GUI or notebooks 03-04 that never went through GoFEM has no resistivities to show yet.

## Settings

In [ ]:
# --- the model -------------------------------------------------------------------------------
MODEL_VTU = "work/04_refine/synthetic/model_synthetic.vtu"   # a GoFEM model (e.g. model_ZK_31.vtu)
RECEIVERS = "work/04_refine/synthetic/receivers_projected.csv"
MAKE_SAMPLE_MODEL = True    # sample only: make the stand-in model if notebook 04 hasn't

# --- putting it on the map ------------------------------------------------------------------
FRAME_FILE = "work/02_survey/model_frame.json"   # the frame it was built in; None: fit it from EDIs
EDI_DIR = "sample_data/edis"   # used when FRAME_FILE is None: matches receiver names to EDI positions
DISPLAY_CRS = "EPSG:32610"     # the maps' projection, e.g. a UTM zone; "" for the model's own frame

# --- the look --------------------------------------------------------------------------------
RHO_RANGE = (1.0, 10000.0)     # colour scale (ohm-m)
CMAP = "turbo_r"               # any matplotlib colour map; "_r" reverses it
DEPTHS_M = [1000, 5000, 15000, 30000]   # plan views, metres below sea level
SECTION_BOTTOM_M = 60000       # how deep the sections go
OUT_DIR = "work/05_figures"

In [ ]:
%matplotlib inline
import os

import matplotlib.pyplot as plt
import numpy as np

from mtinv_gui.backends import projection as P
from mtinv_gui.plotting import project as PP
from mtinv_gui.plotting.backends import georef as G
from mtinv_gui.plotting.backends import render as R
from mtinv_gui.plotting.backends import sections as SEC
from mtinv_gui.plotting.backends.session import PlotSession

os.makedirs(OUT_DIR, exist_ok=True)

## 1. Load the model and put it on the map

The model is in its own frame (metres from the model centre, x = north). Georeferencing says
where that frame sits on the map: read it from notebook 02's frame file, or fit it by matching
the receivers' names to the EDI positions (good to a few cm when the right projection is among
the ones tried).

In [ ]:
if MAKE_SAMPLE_MODEL and not os.path.isfile(MODEL_VTU):
    import sys
    sys.path.insert(0, "sample_data")
    from synthetic_model import make_synthetic_inversion
    make_synthetic_inversion("work/03_mesh", os.path.dirname(MODEL_VTU))

session = PlotSession()
session.load_model(PP.ModelInfo(model_path=MODEL_VTU, receivers_path=RECEIVERS))
print(f"{session.model.n_earth_cells:,} earth cells, {len(session.receivers)} receivers")

if FRAME_FILE:
    frame_georef = G.from_frame(P.load_frame(FRAME_FILE))
    georef = PP.GeorefInfo(method="manual", manual_crs=frame_georef.crs, manual_x0=frame_georef.x0,
                           manual_y0=frame_georef.y0, display_crs=DISPLAY_CRS)
else:
    georef = PP.GeorefInfo(method="stations", station_source=EDI_DIR, display_crs=DISPLAY_CRS)
session.solve_georef(georef)
print("model frame:", P.frame_label(P.load_frame(FRAME_FILE)) if FRAME_FILE else G.crs_label(georef.crs))
if georef.fit_matched:
    print(f"fitted to {georef.fit_matched} stations, {georef.fit_rms_m:.2f} m rms")
print("maps in:", session.maptr.display_label)

A check of the georeferencing: the model's surface elevation under the stations. Coastlines and
valleys should be where you expect them.

In [ ]:
fig = R.new_figure(7, 6)
R.render_station_map(fig, session, labels=True)
display(fig)

## 2. Map layers

Other datasets drawn with the model. Each is a `LayerInfo`; the main settings:

| setting | meaning |
|---|---|
| `kind` | `"raster"` (a grid of values: magnetics, gravity, ...), `"surface"` (a depth grid, e.g. the LAB or basement: also drawn on sections), `"vector"` (shapefile / GeoPackage / GeoJSON / KML outlines), `"points"` (a CSV table) |
| `path` | the file |
| `crs` | the file's coordinate system, if it doesn't say (e.g. `"EPSG:4326"` for longitude / latitude) |
| `map_mode` | rasters: `"image"`, `"contours"` or `"none"`; vectors and points: `"show"` or `"none"` |
| `section_mode` | surfaces: `"line"` or `"profile"`; vectors and points: `"markers"` |
| `color`, `linewidth`, `alpha`, `cmap`, `contour_interval`, `marker`, `size`, `label_col` | the look |

For the example the next cell makes four layers from the topography of notebook 02 (contours, a
coastline, a made-up depth surface) and a table of towns. With your own data, skip it and put
your files' paths in `LAYERS`.

In [ ]:
import warnings

warnings.filterwarnings("ignore", message=".*PyGEOS.*")   # geopandas' notice about its own backend; harmless
import geopandas as gpd
import rasterio
import shapely.geometry as sg
from rasterio.transform import from_origin

from mtinv_gui.backends import topography as T

layer_dir = os.path.join(OUT_DIR, "example_layers")
os.makedirs(layer_dir, exist_ok=True)
dem = T.load_dem_npz("work/02_survey/topo_big.npz")
lat, z = (dem.lat[::-1], dem.z[::-1]) if dem.lat[0] < dem.lat[-1] else (dem.lat, dem.z)   # north-up rows
step = float(abs(dem.lon[1] - dem.lon[0]))
transform = from_origin(dem.lon[0] - step / 2, lat[0] + step / 2, step, step)


def write_tif(path, values):
    with rasterio.open(path, "w", driver="GTiff", width=values.shape[1], height=values.shape[0], count=1,
                       dtype="float32", crs="EPSG:4326", transform=transform, nodata=np.nan) as dst:
        dst.write(values.astype("float32"), 1)
    return path


topo_tif = write_tif(os.path.join(layer_dir, "topography.tif"), z)

# a made-up depth surface (m below sea level): deeper to the south-west, like a dipping horizon
lon_grid, lat_grid = np.meshgrid(dem.lon, lat)
horizon = 12000.0 + 30000.0 * (36.9 - lat_grid) - 20000.0 * (lon_grid + 121.4)
horizon_tif = write_tif(os.path.join(layer_dir, "horizon_depth.tif"), np.clip(horizon, 3000, 40000))

# the coastline: the 0 m contour of the topography, as lines in a GeoJSON file
cs = plt.contour(dem.lon, lat, z, [0.0])
plt.close()
lines = [sg.LineString(seg) for seg in cs.allsegs[0] if len(seg) > 20]
coast_file = os.path.join(layer_dir, "coastline.geojson")
gpd.GeoDataFrame(geometry=lines, crs="EPSG:4326").to_file(coast_file, driver="GeoJSON")

# a table of points
towns_csv = os.path.join(layer_dir, "towns.csv")
with open(towns_csv, "w") as fh:
    fh.write("name,lon,lat\nMonterey,-121.894,36.600\nSalinas,-121.655,36.678\nSanta Cruz,-122.030,36.974\n"
             "Hollister,-121.402,36.853\nKing City,-121.126,36.213\nGilroy,-121.568,37.006\n")
print("example layers in", layer_dir)

In [ ]:
LAYERS = [
    PP.LayerInfo(name="Topography", kind="raster", path=topo_tif, units="m",
                 map_mode="contours", contour_interval=500.0, color="0.35", linewidth=0.5,
                 section_mode="none"),   # "profile" would draw it above each section
    PP.LayerInfo(name="Horizon", kind="surface", path=horizon_tif, surface_ref="depth_bsl", depth_unit="m",
                 map_mode="none", section_mode="line", color="magenta", linewidth=1.5),
    PP.LayerInfo(name="Coastline", kind="vector", path=coast_file, map_mode="show",
                 color="black", linewidth=1.0),
    PP.LayerInfo(name="Towns", kind="points", path=towns_csv, crs="EPSG:4326", x_col="lon", y_col="lat",
                 label_col="name", map_mode="show", section_mode="markers", marker="s", size=20,
                 color="white", label_size=7),
]

## 3. Plan views

One figure per depth. `PlanInfo` holds everything about them (see the table of its settings with
`help(PP.PlanInfo)`); the colour scale is shared through `ColorInfo`.

In [ ]:
colors = PP.ColorInfo(cmap=CMAP, rho_min=RHO_RANGE[0], rho_max=RHO_RANGE[1], n_colors=32)
plan = PP.PlanInfo(depths_m=DEPTHS_M, color=colors,
                   extent=PP.ExtentInfo(mode="stations", padding_m=30000.0),
                   station_labels=False, graticule=True, width_in=7.0, height_in=6.0,
                   title="{depth_label}", dpi=200, file_template="plan_{depth_tag}m.png")

for depth in DEPTHS_M:
    fig = R.new_figure(*R.plan_size(plan, session))
    R.render_plan(fig, session, plan, depth, LAYERS)
    display(fig)

Adding your own drawing: `render_plan` returns the map's axes, in the display CRS (metres). Here a
marker and a label at a longitude / latitude:

In [ ]:
fig = R.new_figure(*R.plan_size(plan, session))
ax = R.render_plan(fig, session, plan, 15000.0, LAYERS)
e, n = session.maptr.from_lonlat(np.array([-121.5]), np.array([36.75]))
ax.plot(e, n, "*", ms=18, mfc="yellow", mec="k", zorder=20)
ax.annotate("a point of interest", (e[0], n[0]), xytext=(8, 8), textcoords="offset points",
            fontsize=9, zorder=20)
display(fig)

## 4. Sections

A section is a `SectionInfo`:
* `kind="ew"` / `"ns"`: a straight east-west (north-south) line at `coord_m`, a northing (easting)
  in the display CRS. With `start_m == end_m` it spans the stations.
* `kind="polyline"`: through `vertices=[[e, n], ...]` in the display CRS.
* `kind="stations"`: through the named stations, in order (`extend_m` past each end).

All sections share one `SectionStyle` (depth range, colours, vertical exaggeration via each
section's `ve`), so a set of them matches.

In [ ]:
st = session.stations_display()                      # station (easting, northing) in the display CRS
centre_e, centre_n = 0.5 * (st[:, 0].min() + st[:, 0].max()), 0.5 * (st[:, 1].min() + st[:, 1].max())
names = session.receivers.names

style = PP.SectionStyle(depth_bottom_m=SECTION_BOTTOM_M, color=colors, show_stations=True,
                        station_labels=True, station_max_offset_m=11000.0, topo_profile=True,
                        width_in=10.0, dpi=200, file_template="section_{label}.png")
SECTIONS = [
    PP.SectionInfo(label="A", kind="ew", coord_m=centre_n, ve=2.0),
    PP.SectionInfo(label="B", kind="ns", coord_m=centre_e + 10000.0, ve=2.0),
    PP.SectionInfo(label="C", kind="stations", stations=["CST05", "CST13", "CST20", "CST31"],
                   extend_m=10000.0, ve=2.0),
]

for sec in SECTIONS:
    data = SEC.prepare(session, sec, style, LAYERS)
    fig = R.new_figure(*R.section_size(sec, style, data, session))
    R.render_section(fig, session, sec, style, data)
    display(fig)

Where the sections run:

In [ ]:
fig = R.new_figure(7, 6)
R.render_index_map(fig, session, SECTIONS, style, plan, layers=LAYERS)
display(fig)

## 5. Save the figures

The format follows the file name: `.png`, `.pdf`, `.svg`, ... (`plan.file_template`,
`style.file_template`).

In [ ]:
written = R.export_plan_all(OUT_DIR, session, plan, layers=LAYERS)
for sec in SECTIONS:
    written.append(R.export_section(os.path.join(OUT_DIR, R.section_file_name(sec, style, session)),
                                    session, sec, style, layers=LAYERS))
fig = R.new_figure(7, 6)
R.render_index_map(fig, session, SECTIONS, style, plan, layers=LAYERS)
written.append(R.save_figure(fig, os.path.join(OUT_DIR, "section_map.png"), session, 200))
print("\n".join(written))

## Next

Notebook `06_model_to_geotiff` writes the depth slices as GeoTIFFs for GIS.

**For coders:** `render_plan` / `render_section` draw from `plotting/backends/slicing.py`
(`plan_slice`: the cells at a depth as polygons) and `sections.py` (`cut_section`). For a figure
these don't cover, get the polygons and values from those and draw them yourself; the plotting
GUI's *Figures* tab and `mtplot-batch` reuse the same functions for multi-panel layouts.